# ChromaDB Embedding Space Visualization

This notebook provides interactive visualizations of your embedding space including:
- 2D/3D projections using t-SNE and UMAP
- Distance distributions
- Cluster analysis
- Nearest neighbor exploration
- Metadata correlations

## Setup and Installation

In [ ]:
# Install required packages (run this cell first if packages are not installed)
# !pip install chromadb numpy pandas matplotlib seaborn plotly scikit-learn umap-learn

In [ ]:
import chromadb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✅ All packages imported successfully!")

## 1. Load Data from ChromaDB using Python Client

In [ ]:
import chromadb

# Path to the directory containing chroma.sqlite3
chroma_path = "../app_colette/mm_index"  # Change this to your ChromaDB directory

# Connect to ChromaDB
client = chromadb.PersistentClient(path=chroma_path)

# List all collections
collections = client.list_collections()
print(f"Found {len(collections)} collection(s):")
for col in collections:
    print(f"  - {col.name} (count: {col.count()})")

In [ ]:
# Select a collection to visualize
# Change collection_index to select different collections (0 = first collection)
collection_index = 0
collection_name = collections[collection_index].name
collection = client.get_collection(collection_name)

print(f"Selected collection: {collection_name}")
print(f"Total documents: {collection.count()}")

In [ ]:
# Get all data from the collection
print("Loading embeddings from ChromaDB...")
results = collection.get(
    include=['embeddings', 'documents', 'metadatas']
)

embeddings = np.array(results['embeddings'])
documents = results['documents']
metadatas = results['metadatas']
ids = results['ids']

print(f"\n📊 Data Summary:")
print(f"  - Number of embeddings: {len(embeddings)}")
print(f"  - Embedding dimension: {embeddings.shape[1]}")
print(f"  - Document samples: {len(documents)}")
print(f"  - Metadata entries: {len(metadatas)}")

# Verify embeddings are valid
if len(embeddings) > 0:
    print(f"\n✓ Embedding statistics:")
    print(f"  - Mean norm: {np.mean([np.linalg.norm(e) for e in embeddings]):.4f}")
    print(f"  - Mean value: {np.mean(embeddings):.6f}")
    print(f"  - Std value: {np.std(embeddings):.6f}")

# Create a DataFrame for easier analysis
df = pd.DataFrame({
    'id': ids,
    'document': documents,
    'document_preview': [str(doc)[:100] + '...' if len(str(doc)) > 100 else str(doc) for doc in documents]
})

# Add metadata columns
if metadatas and any(metadatas):
    metadata_df = pd.DataFrame(metadatas)
    df = pd.concat([df, metadata_df], axis=1)

print(f"\nSample metadata keys: {list(metadata_df.columns) if 'metadata_df' in locals() and not metadata_df.empty else 'None'}")
df.head()

## 2. Basic Statistics and Distance Analysis

In [ ]:
# Compute pairwise distances
print("Computing pairwise distances...")
distances = pdist(embeddings, metric='cosine')
distance_matrix = squareform(distances)

print(f"\n📏 Distance Statistics:")
print(f"  - Mean distance: {np.mean(distances):.4f}")
print(f"  - Median distance: {np.median(distances):.4f}")
print(f"  - Std deviation: {np.std(distances):.4f}")
print(f"  - Min distance: {np.min(distances):.4f}")
print(f"  - Max distance: {np.max(distances):.4f}")

In [ ]:
# Visualize distance distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(distances, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(np.mean(distances), color='red', linestyle='--', label=f'Mean: {np.mean(distances):.3f}')
axes[0].axvline(np.median(distances), color='green', linestyle='--', label=f'Median: {np.median(distances):.3f}')
axes[0].set_xlabel('Cosine Distance')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Distribution of Pairwise Distances')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Box plot
axes[1].boxplot(distances, vert=True)
axes[1].set_ylabel('Cosine Distance')
axes[1].set_title('Distance Distribution (Box Plot)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap of distance matrix (sample if too large)
sample_size = min(50, len(embeddings))
sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)
sample_distances = distance_matrix[np.ix_(sample_indices, sample_indices)]

plt.figure(figsize=(12, 10))
sns.heatmap(sample_distances, cmap='viridis', square=True, 
            xticklabels=False, yticklabels=False,
            cbar_kws={'label': 'Cosine Distance'})
plt.title(f'Distance Matrix Heatmap (Sample of {sample_size} documents)')
plt.xlabel('Document Index')
plt.ylabel('Document Index')
plt.tight_layout()
plt.show()

## 3. Dimensionality Reduction: PCA

In [ ]:
# PCA to understand variance
print("Performing PCA...")
pca_full = PCA(n_components=min(48, embeddings.shape[1]))
pca_full.fit(embeddings)

# Plot explained variance
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Scree plot
axes[0].plot(range(1, len(pca_full.explained_variance_ratio_) + 1), 
             pca_full.explained_variance_ratio_, 'o-')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot')
axes[0].grid(True, alpha=0.3)

# Cumulative variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)
axes[1].plot(range(1, len(cumsum) + 1), cumsum, 'o-')
axes[1].axhline(y=0.95, color='r', linestyle='--', label='95% variance')
axes[1].set_xlabel('Number of Components')
axes[1].set_ylabel('Cumulative Explained Variance')
axes[1].set_title('Cumulative Explained Variance')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

n_components_95 = np.argmax(cumsum >= 0.95) + 1
print(f"\n📊 Components needed for 95% variance: {n_components_95}")

In [ ]:
# PCA 2D projection
print("Computing 2D PCA projection...")
pca_2d = PCA(n_components=2)
embeddings_pca_2d = pca_2d.fit_transform(embeddings)

df['pca_x'] = embeddings_pca_2d[:, 0]
df['pca_y'] = embeddings_pca_2d[:, 1]

# Interactive 2D PCA plot
fig = px.scatter(df, x='pca_x', y='pca_y', 
                 hover_data=['document_preview'],
                 title='2D PCA Projection of Embeddings',
                 labels={'pca_x': f'PC1 ({pca_2d.explained_variance_ratio_[0]:.2%} var)',
                        'pca_y': f'PC2 ({pca_2d.explained_variance_ratio_[1]:.2%} var)'})
fig.update_traces(marker=dict(size=8, opacity=0.6))
fig.show()

In [ ]:
# PCA 3D projection
print("Computing 3D PCA projection...")
pca_3d = PCA(n_components=3)
embeddings_pca_3d = pca_3d.fit_transform(embeddings)

df['pca_z'] = embeddings_pca_3d[:, 2]

# Interactive 3D PCA plot
fig = px.scatter_3d(df, x='pca_x', y='pca_y', z='pca_z',
                    hover_data=['document_preview'],
                    title='3D PCA Projection of Embeddings',
                    labels={'pca_x': f'PC1 ({pca_3d.explained_variance_ratio_[0]:.2%})',
                           'pca_y': f'PC2 ({pca_3d.explained_variance_ratio_[1]:.2%})',
                           'pca_z': f'PC3 ({pca_3d.explained_variance_ratio_[2]:.2%})'})
fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.show()

## 4. t-SNE Visualization

In [ ]:
# t-SNE 2D
print("Computing t-SNE 2D projection (this may take a while)...")
tsne_2d = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
embeddings_tsne_2d = tsne_2d.fit_transform(embeddings)

df['tsne_x'] = embeddings_tsne_2d[:, 0]
df['tsne_y'] = embeddings_tsne_2d[:, 1]

# Interactive 2D t-SNE plot
fig = px.scatter(df, x='tsne_x', y='tsne_y',
                 hover_data=['document_preview'],
                 title='2D t-SNE Projection of Embeddings',
                 labels={'tsne_x': 't-SNE Dimension 1', 'tsne_y': 't-SNE Dimension 2'})
fig.update_traces(marker=dict(size=8, opacity=0.6))
fig.show()

print("✅ t-SNE 2D complete!")

In [ ]:
# t-SNE 3D
print("Computing t-SNE 3D projection (this may take a while)...")
tsne_3d = TSNE(n_components=3, random_state=42, perplexity=min(30, len(embeddings)-1))
embeddings_tsne_3d = tsne_3d.fit_transform(embeddings)

df['tsne_z'] = embeddings_tsne_3d[:, 2]

# Interactive 3D t-SNE plot
fig = px.scatter_3d(df, x='tsne_x', y='tsne_y', z='tsne_z',
                    hover_data=['document_preview'],
                    title='3D t-SNE Projection of Embeddings',
                    labels={'tsne_x': 't-SNE Dim 1', 'tsne_y': 't-SNE Dim 2', 'tsne_z': 't-SNE Dim 3'})
fig.update_traces(marker=dict(size=5, opacity=0.6))
fig.show()

print("✅ t-SNE 3D complete!")

## 5. UMAP Visualization (Better than t-SNE for large datasets)

In [ ]:
try:
    import umap
    
    # UMAP 2D
    print("Computing UMAP 2D projection...")
    reducer_2d = umap.UMAP(n_components=2, random_state=42, n_neighbors=min(15, len(embeddings)-1))
    embeddings_umap_2d = reducer_2d.fit_transform(embeddings)
    
    df['umap_x'] = embeddings_umap_2d[:, 0]
    df['umap_y'] = embeddings_umap_2d[:, 1]
    
    # Interactive 2D UMAP plot
    fig = px.scatter(df, x='umap_x', y='umap_y',
                     hover_data=['document_preview'],
                     title='2D UMAP Projection of Embeddings',
                     labels={'umap_x': 'UMAP Dimension 1', 'umap_y': 'UMAP Dimension 2'})
    fig.update_traces(marker=dict(size=8, opacity=0.6))
    fig.show()
    
    print("✅ UMAP 2D complete!")
    
except ImportError:
    print("⚠️ UMAP not installed. Run: pip install umap-learn")

In [ ]:
try:
    # UMAP 3D
    print("Computing UMAP 3D projection...")
    reducer_3d = umap.UMAP(n_components=3, random_state=42, n_neighbors=min(15, len(embeddings)-1))
    embeddings_umap_3d = reducer_3d.fit_transform(embeddings)
    
    df['umap_z'] = embeddings_umap_3d[:, 2]
    
    # Interactive 3D UMAP plot
    fig = px.scatter_3d(df, x='umap_x', y='umap_y', z='umap_z',
                        hover_data=['document_preview'],
                        title='3D UMAP Projection of Embeddings',
                        labels={'umap_x': 'UMAP Dim 1', 'umap_y': 'UMAP Dim 2', 'umap_z': 'UMAP Dim 3'})
    fig.update_traces(marker=dict(size=5, opacity=0.6))
    fig.show()
    
    print("✅ UMAP 3D complete!")
    
except (ImportError, NameError):
    print("⚠️ UMAP not available")

## 6. Clustering Analysis

In [ ]:
# Find optimal number of clusters using elbow method
print("Finding optimal number of clusters...")
max_clusters = min(10, len(embeddings) - 1)
inertias = []
silhouette_scores = []
K_range = range(2, max_clusters + 1)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(embeddings)
    inertias.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(embeddings, kmeans.labels_))

# Plot elbow curve and silhouette scores
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].plot(K_range, inertias, 'o-')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method for Optimal k')
axes[0].grid(True, alpha=0.3)

axes[1].plot(K_range, silhouette_scores, 'o-', color='green')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs k')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

optimal_k = K_range[np.argmax(silhouette_scores)]
print(f"\n📊 Optimal number of clusters (by silhouette score): {optimal_k}")

In [ ]:
# Apply K-Means clustering
n_clusters = optimal_k
print(f"Performing K-Means clustering with k={n_clusters}...")
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(embeddings)
df['cluster'] = df['cluster'].astype(str)

print(f"\n📊 Cluster Distribution:")
print(df['cluster'].value_counts().sort_index())

In [ ]:
# Visualize clusters in 2D (using t-SNE or UMAP)
# Using t-SNE
fig = px.scatter(df, x='tsne_x', y='tsne_y', color='cluster',
                 hover_data=['document_preview'],
                 title=f't-SNE Projection with K-Means Clusters (k={n_clusters})',
                 labels={'tsne_x': 't-SNE Dimension 1', 'tsne_y': 't-SNE Dimension 2'})
fig.update_traces(marker=dict(size=8, opacity=0.7))
fig.show()

In [ ]:
# 3D cluster visualization
fig = px.scatter_3d(df, x='tsne_x', y='tsne_y', z='tsne_z', color='cluster',
                    hover_data=['document_preview'],
                    title=f'3D t-SNE Projection with K-Means Clusters (k={n_clusters})',
                    labels={'tsne_x': 't-SNE Dim 1', 'tsne_y': 't-SNE Dim 2', 'tsne_z': 't-SNE Dim 3'})
fig.update_traces(marker=dict(size=5, opacity=0.7))
fig.show()

In [ ]:
# Analyze cluster characteristics
print("Cluster Characteristics:\n")
for cluster_id in sorted(df['cluster'].unique()):
    cluster_docs = df[df['cluster'] == cluster_id]
    print(f"\n{'='*60}")
    print(f"Cluster {cluster_id} - {len(cluster_docs)} documents")
    print(f"{'='*60}")
    print("\nSample documents:")
    for i, doc in enumerate(cluster_docs['document_preview'].head(3)):
        print(f"  {i+1}. {doc}")
    
    # If metadata exists, show common metadata values
    if metadatas and metadatas[0]:
        print("\nMetadata distribution:")
        for col in metadata_df.columns:
            if col in cluster_docs.columns:
                print(f"  {col}: {cluster_docs[col].value_counts().head(3).to_dict()}")

## 7. Nearest Neighbors Analysis

In [ ]:
# Function to find and visualize nearest neighbors
def visualize_nearest_neighbors(doc_index, n_neighbors=5):
    """
    Find and visualize nearest neighbors for a given document
    """
    # Get distances from the selected document
    distances_from_doc = distance_matrix[doc_index]
    
    # Find nearest neighbors (excluding itself)
    nearest_indices = np.argsort(distances_from_doc)[1:n_neighbors+1]
    
    print(f"\n{'='*80}")
    print(f"Document {doc_index}: {df.iloc[doc_index]['document_preview']}")
    print(f"{'='*80}\n")
    print(f"Top {n_neighbors} Nearest Neighbors:\n")
    
    for i, idx in enumerate(nearest_indices, 1):
        print(f"{i}. Distance: {distances_from_doc[idx]:.4f}")
        print(f"   Doc: {df.iloc[idx]['document_preview']}\n")
    
    # Visualize in t-SNE space
    highlight_indices = [doc_index] + list(nearest_indices)
    df_viz = df.copy()
    df_viz['highlight'] = 'Other'
    df_viz.loc[doc_index, 'highlight'] = 'Query Document'
    df_viz.loc[nearest_indices, 'highlight'] = 'Nearest Neighbor'
    
    fig = px.scatter(df_viz, x='tsne_x', y='tsne_y', color='highlight',
                     hover_data=['document_preview'],
                     title=f'Nearest Neighbors in t-SNE Space',
                     color_discrete_map={'Query Document': 'red', 
                                        'Nearest Neighbor': 'green', 
                                        'Other': 'lightgray'})
    fig.update_traces(marker=dict(size=8, opacity=0.7))
    fig.show()

# Example: Find neighbors for the first document
visualize_nearest_neighbors(0, n_neighbors=5)

In [ ]:
# Interactive widget to explore neighbors
print("You can explore neighbors for different documents by calling:")
print("visualize_nearest_neighbors(doc_index, n_neighbors=5)")
print(f"\nValid doc_index range: 0 to {len(df)-1}")

## 8. Hierarchical Clustering Dendrogram

In [ ]:
# Create dendrogram (sample if dataset is large)
sample_size = min(100, len(embeddings))
if len(embeddings) > sample_size:
    print(f"Sampling {sample_size} documents for dendrogram...")
    sample_indices = np.random.choice(len(embeddings), sample_size, replace=False)
    sample_embeddings = embeddings[sample_indices]
else:
    sample_embeddings = embeddings

print("Computing hierarchical clustering...")
linkage_matrix = linkage(sample_embeddings, method='ward')

plt.figure(figsize=(15, 8))
dendrogram(linkage_matrix, no_labels=True)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Document Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

## 9. Density Analysis

In [ ]:
# 2D density plot using t-SNE coordinates
from scipy.stats import gaussian_kde

# Calculate density
xy = np.vstack([df['tsne_x'], df['tsne_y']])
z = gaussian_kde(xy)(xy)

df['density'] = z

fig = px.scatter(df, x='tsne_x', y='tsne_y', color='density',
                 hover_data=['document_preview'],
                 title='Embedding Space Density (t-SNE)',
                 color_continuous_scale='Viridis',
                 labels={'tsne_x': 't-SNE Dimension 1', 'tsne_y': 't-SNE Dimension 2'})
fig.update_traces(marker=dict(size=8, opacity=0.6))
fig.show()

In [ ]:
# Hexbin density plot
fig, ax = plt.subplots(figsize=(12, 8))
hexbin = ax.hexbin(df['tsne_x'], df['tsne_y'], gridsize=30, cmap='YlOrRd', mincnt=1)
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.set_title('Embedding Space Density (Hexbin)')
plt.colorbar(hexbin, ax=ax, label='Count')
plt.tight_layout()
plt.show()

## 10. Export Results

In [ ]:
# # Save the enriched dataframe
# output_file = f'{collection_name}_embeddings_analysis.csv'
# df.to_csv(output_file, index=False)
# print(f"✅ Results saved to: {output_file}")

# # Save embeddings and projections
# np.savez(f'{collection_name}_projections.npz',
#          embeddings=embeddings,
#          pca_2d=embeddings_pca_2d,
#          pca_3d=embeddings_pca_3d,
#          tsne_2d=embeddings_tsne_2d,
#          tsne_3d=embeddings_tsne_3d,
#          distance_matrix=distance_matrix)
# print(f"✅ Projections saved to: {collection_name}_projections.npz")

## 11. Summary Statistics

In [ ]:
print("\n" + "="*80)
print("EMBEDDING SPACE ANALYSIS SUMMARY")
print("="*80)
print(f"\n📊 Dataset: {collection_name}")
print(f"   - Total documents: {len(embeddings)}")
print(f"   - Embedding dimension: {embeddings.shape[1]}")
print(f"\n📏 Distance Statistics:")
print(f"   - Mean distance: {np.mean(distances):.4f}")
print(f"   - Median distance: {np.median(distances):.4f}")
print(f"   - Std deviation: {np.std(distances):.4f}")
print(f"\n🎯 Clustering:")
print(f"   - Optimal clusters (K-Means): {n_clusters}")
print(f"   - Silhouette score: {silhouette_score(embeddings, kmeans.labels_):.4f}")
print(f"\n📐 Dimensionality Reduction:")
print(f"   - PCA components for 95% variance: {n_components_95}")
print(f"   - PCA explained variance (first 3): {pca_3d.explained_variance_ratio_.sum():.2%}")
print("\n" + "="*80)